# Lecture 6 — Regularization, sparsity, and robustness

**Week 2 · Day 6 · 45 min**

> **Headline.** Ridge was smooth. **Lasso is not** — and that non-smoothness is not an
> obstacle to work around. It is exactly what produces sparsity.

Every method this week has assumed a differentiable objective. Today the penalty has a
kink at zero, gradient methods break on it, and the repair — the proximal operator — is
both simple and the gateway to a large modern family of algorithms.

**By the end of this lecture you can:**

1. explain why $\|w\|_1$ pins coefficients to *exactly* zero while $\|w\|_2^2$ does not;
2. define a proximal operator and derive the soft-threshold;
3. write the ISTA step and say what FISTA adds;
4. explain why Huber is robust *and* smooth, and how that differs from L1.

**You implement this afternoon:** `L1`, `ElasticNet`, `ProximalGradient`, `Huber`,
`PoissonNLL`, plus the benchmark and the extensibility challenge.

### Pacing

Target **37 min** of core material, hard cap **45 min**. Sections marked
*(cut first)* are the ones to drop if you are running behind; everything else is
load-bearing for the labwork. **The times below already include showing and discussing
the figures** — each figure is produced by the code cell above it, so run the notebook
once before the session.


> Figure 1(c) — the diamond and the circle — is the single most memorable image of the
> day; show it even if you show nothing else. Figure 4 (FISTA's rate) is the one to cut.


| § | Section | min |
|---|---|---|
| 1 | Why gradient methods fail on $\|w\|_1$  — *Figure 1* | 8 |
| 2 | The proximal operator  — *Figure 2* | 9 |
| 3 | Proximal gradient (ISTA)  — *Figures 3–4* | 10 |
| 4 | Robustness: the Huber loss  — *Figure 5* | 8 |
| 5 | The week, in one diagram  *(cut first)* | 4 |
| 6 | Today's labs | 2 |
| | **total** | **41** |
| | **core only** | **37** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# One consistent look for every figure in the lecture.
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 9,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# If this import fails:   pip install matplotlib

rng = np.random.default_rng(6)

---

## 1. Why gradient methods fail on $\|w\|_1$

$$\min_w \; \underbrace{f(w)}_{\text{smooth}} \;+\; \underbrace{\lambda\|w\|_1}_{\text{not differentiable at } 0}$$

$|w_j|$ has no derivative at $w_j = 0$: the slope is $-1$ on the left and $+1$ on the
right. And the solution *wants* to sit exactly there — that is the entire purpose.

So the objective is non-differentiable precisely at the points we are trying to reach.
Gradient descent will oscillate across zero and never land on it; worse, on floating-point
arithmetic you get coefficients like $10^{-9}$ rather than $0$, so you do not actually get
a sparse model — just a small one.

### Why the kink creates sparsity

Take one coordinate and compare the two penalties near zero.

| | penalty | derivative as $w \to 0^+$ |
|---|---|---|
| ridge | $\tfrac{\lambda}{2}w^2$ | $\lambda w \to 0$ |
| lasso | $\lambda|w|$ | $\lambda$ — **constant** |

Ridge's pull toward zero *vanishes* as you approach zero, so a coordinate drifts closer
and closer but never arrives. Lasso's pull stays at full strength $\lambda$ all the way
in. If the data's pull on that coordinate is weaker than $\lambda$, the penalty wins
outright and the coefficient is set to **exactly** zero and stays there.

That is the whole mechanism: **a constant restoring force beats a vanishing one.**
Sparsity, feature selection, and the Laplace prior of day 1 are three descriptions of it.

In [ ]:
# Why the kink matters: the penalty, its slope, and the geometry it creates.
fig, ax = plt.subplots(1, 3, figsize=(14, 4.0))
lam = 1.0
ws = np.linspace(-2, 2, 800)

# --- (a) the two penalties.
ax[0].plot(ws, 0.5 * lam * ws ** 2, lw=2.4, color="tab:blue",
           label=r"ridge  $\frac{\lambda}{2}w^2$")
ax[0].plot(ws, lam * np.abs(ws), lw=2.4, color="crimson", label=r"lasso  $\lambda|w|$")
ax[0].plot(0, 0, "o", color="crimson", ms=9, zorder=6)
ax[0].annotate("the kink", xy=(0, 0), xytext=(0, 42), textcoords="offset points",
               fontsize=9, color="crimson", ha="center", fontweight="bold",
               arrowprops=dict(arrowstyle="->", color="crimson", lw=1.3))
ax[0].set_xlabel("$w$"); ax[0].set_ylabel("penalty")
ax[0].set_title("The penalties", fontsize=9); ax[0].legend(fontsize=8.5)

# --- (b) their SLOPES: this is where the mechanism actually is.
ax[1].plot(ws, lam * ws, lw=2.4, color="tab:blue", label=r"ridge: $\lambda w$")
ax[1].plot(ws[ws < -1e-9], -lam * np.ones((ws < -1e-9).sum()), lw=2.4, color="crimson")
ax[1].plot(ws[ws > 1e-9], lam * np.ones((ws > 1e-9).sum()), lw=2.4, color="crimson",
           label=r"lasso: $\lambda\,\mathrm{sign}(w)$")
ax[1].plot([0, 0], [-lam, lam], ":", lw=2.0, color="crimson")
ax[1].plot(0, 0, "o", mfc="white", mec="tab:blue", mew=2.0, ms=9, zorder=6)
ax[1].annotate("ridge's pull VANISHES\nas $w \\to 0$", xy=(0.0, 0.0), xytext=(24, -48),
               textcoords="offset points", fontsize=8, color="tab:blue",
               bbox=dict(fc="white", ec="tab:blue", lw=0.8, alpha=0.95),
               arrowprops=dict(arrowstyle="->", color="tab:blue", lw=1.1))
ax[1].annotate("lasso's pull stays at $\\lambda$\nall the way in", xy=(0.55, lam),
               xytext=(-14, 30), textcoords="offset points", fontsize=8, color="crimson",
               bbox=dict(fc="white", ec="crimson", lw=0.8, alpha=0.95),
               arrowprops=dict(arrowstyle="->", color="crimson", lw=1.1))
ax[1].axhline(0, color="0.5", lw=0.8)
ax[1].set_xlabel("$w$"); ax[1].set_ylabel("derivative of the penalty")
ax[1].set_ylim(-2.2, 2.2)
ax[1].set_title("The slopes — the mechanism lives here", fontsize=9)
ax[1].legend(fontsize=8.5, loc="lower right")

# --- (c) the classic geometry, in two dimensions.
w_ls = np.array([2.1, 0.75])                      # unconstrained least-squares solution
Q = np.array([[1.0, 0.72], [0.72, 0.62]])         # shape of the data's level sets
g1, g2 = np.meshgrid(np.linspace(-1.7, 3.2, 400), np.linspace(-1.6, 2.0, 400))
D1, D2 = g1 - w_ls[0], g2 - w_ls[1]
Z = Q[0, 0] * D1 ** 2 + 2 * Q[0, 1] * D1 * D2 + Q[1, 1] * D2 ** 2
ax[2].contour(g1, g2, Z, levels=np.geomspace(0.06, 9, 11), colors="0.68", linewidths=0.9)

tt = 1.0
ax[2].plot([tt, 0, -tt, 0, tt], [0, tt, 0, -tt, 0], lw=2.4, color="crimson",
           label=r"$\|w\|_1 \leq t$")
th = np.linspace(0, 2 * np.pi, 300)
ax[2].plot(tt * np.cos(th), tt * np.sin(th), lw=2.4, color="tab:blue",
           label=r"$\|w\|_2 \leq t$")

# Where each constrained solution actually lands.
cand = np.array([[tt, 0], [0, tt], [-tt, 0], [0, -tt]])
edge = np.linspace(0, 1, 4001)
pts = np.vstack([np.outer(1 - edge, cand[i]) + np.outer(edge, cand[(i + 1) % 4])
                 for i in range(4)])
q = lambda P: (Q[0, 0] * (P[:, 0] - w_ls[0]) ** 2
               + 2 * Q[0, 1] * (P[:, 0] - w_ls[0]) * (P[:, 1] - w_ls[1])
               + Q[1, 1] * (P[:, 1] - w_ls[1]) ** 2)
p_l1 = pts[np.argmin(q(pts))]
circ = np.column_stack([tt * np.cos(th), tt * np.sin(th)])
p_l2 = circ[np.argmin(q(circ))]

ax[2].plot(*w_ls, "k+", ms=14, mew=2.4, zorder=7)
ax[2].annotate("unconstrained\nleast squares", xy=w_ls, xytext=(-4, 14),
               textcoords="offset points", fontsize=7.5, ha="center")
ax[2].plot(*p_l1, "o", color="crimson", ms=12, mec="k", mew=0.9, zorder=8,
           label=f"lasso: CORNER, $w_2$ = {p_l1[1]:.2f}")
ax[2].plot(*p_l2, "o", color="tab:blue", ms=12, mec="k", mew=0.9, zorder=8,
           label=f"ridge: ({p_l2[0]:.2f}, {p_l2[1]:.2f})")
ax[2].set_aspect("equal"); ax[2].set_xlabel("$w_1$"); ax[2].set_ylabel("$w_2$")
ax[2].set_title("Why corners give exact zeros", fontsize=9)
ax[2].set_xlim(-1.7, 3.2); ax[2].set_ylim(-1.6, 2.0)
ax[2].legend(fontsize=7.5, loc="lower left", framealpha=0.95)

plt.tight_layout()
plt.show()

print(f"lasso solution: ({p_l1[0]:.4f}, {p_l1[1]:.4f})   -> w2 is EXACTLY zero")
print(f"ridge solution: ({p_l2[0]:.4f}, {p_l2[1]:.4f})   -> both coordinates non-zero")

**Figure 1 — three views of the same fact.**

*(a)* The two penalties. Ridge is a smooth parabola; lasso has a **kink** at zero. So far
this looks like a technicality about differentiability.

*(b)* Their derivatives, where the mechanism actually lives. Ridge's restoring force is
$\lambda w$, which **fades to nothing** as the coefficient approaches zero — so a
coefficient creeps toward zero ever more slowly and never arrives. Lasso's force is
$\lambda\,\mathrm{sign}(w)$: full strength $\lambda$ right up to the boundary, then an
abrupt jump. If the data pulls on that coefficient with less force than $\lambda$, the
penalty wins outright and the coefficient is *pinned* at exactly zero.

*(c)* The same thing geometrically. Minimizing subject to a budget on the penalty means
inflating the grey ellipses until they first touch the constraint region. The $\ell_1$
ball is a **diamond**: it has corners, and the corners lie on the axes — where a
coordinate is exactly zero. A stretched ellipse almost always touches a corner first, so
the solution is sparse. The $\ell_2$ ball is a circle, with no special points, so the
touch happens at a generic spot where both coordinates are non-zero but smaller.

> **Sparsity is not a numerical accident and it is not thresholding after the fact.** It
> is where the optimum genuinely is, and that is why we are willing to build a whole new
> algorithm (§2–§3) to reach a point the methods of days 2–5 cannot land on.

---

## 2. The proximal operator

Since we cannot differentiate $r$, we solve a small problem involving it exactly instead.

$$\boxed{\;\mathrm{prox}_{t r}(v) \;=\; \arg\min_{w} \; \tfrac{1}{2}\|w - v\|^2 + t\,r(w)\;}$$

In words: *"move toward $v$, but pay $t \cdot r$ for where you end up."* It balances
staying near $v$ against the penalty. Note it is always well defined for convex $r$ — the
objective is strongly convex — even where $r$ has no gradient.

### The soft-threshold

For $r(w) = \lambda\|w\|_1$ the problem separates across coordinates, so take one:

$$\min_w \; \tfrac{1}{2}(w - v)^2 + t\lambda|w|$$

For $w > 0$: derivative $w - v + t\lambda = 0 \Rightarrow w = v - t\lambda$, valid when $v > t\lambda$.
For $w < 0$: derivative $w - v - t\lambda = 0 \Rightarrow w = v + t\lambda$, valid when $v < -t\lambda$.
Otherwise the minimum is at the kink, $w = 0$. Together:

$$\mathrm{prox}_{t\lambda\|\cdot\|_1}(v) = \mathrm{sign}(v)\,\max(|v| - t\lambda,\, 0) \;=:\; S_{t\lambda}(v)$$

Shrink every coordinate toward zero by $t\lambda$, and **clamp at zero** rather than
crossing it. The clamping is where the exact zeros come from.

Compare the three we care about:

| $r(w)$ | $\mathrm{prox}_{tr}(v)$ | effect |
|---|---|---|
| $0$ | $v$ | identity |
| $\tfrac{\lambda}{2}\|w\|_2^2$ | $v/(1 + t\lambda)$ | shrink proportionally — **never exactly 0** |
| $\lambda\|w\|_1$ | $\mathrm{sign}(v)\max(|v| - t\lambda, 0)$ | shrink by a constant — **exactly 0 inside the band** |

In [ ]:
def soft_threshold(v, t):
    return np.sign(v) * np.maximum(np.abs(v) - t, 0.0)


v = np.array([3.0, -0.4, 0.1])
print(f"v            = {v}")
print(f"soft(v, 0.5) = {soft_threshold(v, 0.5)}   <- hand result: (2.5, 0, 0)")
print(f"ridge prox   = {v / (1 + 0.5)}   <- shrunk, but nothing is exactly zero")

print("\nThe L1 prox zeroed 2 of 3 coordinates. The ridge prox zeroed none.")
print("That is feature selection versus mere shrinkage, in one line of arithmetic.")

In [ ]:
# The 1-D lasso, solved in closed form: min 0.5(w-3)^2 + lam|w|  =>  w = soft(3, lam)
print("min 0.5*(w - 3)^2 + lam*|w|\n")
print(f"{'lam':>6} {'w (closed form)':>18} {'w (grid search)':>18}")
grid = np.linspace(-1, 4, 500001)
for lam in [0.0, 0.5, 1.0, 2.0, 3.0, 4.0]:
    closed = float(soft_threshold(np.array([3.0]), lam)[0])
    brute = grid[np.argmin(0.5 * (grid - 3) ** 2 + lam * np.abs(grid))]
    print(f"{lam:6.1f} {closed:18.4f} {brute:18.4f}")
print("\nPast lam = 3 the penalty wins outright and the coefficient is exactly 0.")

In [ ]:
# The prox operators themselves: input on the x axis, output on the y axis.
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3))
vv = np.linspace(-3, 3, 800)
t_lam = 1.0

ax[0].plot(vv, vv, ":", lw=1.4, color="0.5", label="identity  ($r = 0$)")
ax[0].plot(vv, vv / (1 + t_lam), lw=2.4, color="tab:blue",
           label=r"ridge prox  $v/(1+t\lambda)$")
ax[0].plot(vv, soft_threshold(vv, t_lam), lw=2.6, color="crimson",
           label=r"soft-threshold  $S_{t\lambda}(v)$")
ax[0].axvspan(-t_lam, t_lam, color="crimson", alpha=0.10, zorder=0)
ax[0].annotate("the dead zone:\neverything here\nmaps to EXACTLY 0",
               xy=(0, 0), xytext=(0, -78), textcoords="offset points", fontsize=8,
               color="crimson", ha="center",
               bbox=dict(fc="white", ec="crimson", lw=0.9, alpha=0.95),
               arrowprops=dict(arrowstyle="->", color="crimson", lw=1.2))
ax[0].axhline(0, color="0.6", lw=0.8); ax[0].axvline(0, color="0.6", lw=0.8)
ax[0].set_xlabel("input $v$"); ax[0].set_ylabel("output")
ax[0].set_title(f"The prox maps, at $t\\lambda$ = {t_lam}", fontsize=9)
ax[0].legend(fontsize=8, loc="upper left"); ax[0].set_aspect("equal")

# --- right: the 1-D objective the prox is silently minimizing.
ws2 = np.linspace(-1.0, 4.0, 900)
v0 = 3.0
for lm, col in [(0.0, "0.55"), (1.0, "tab:olive"), (3.0, "tab:orange"),
                (4.5, "crimson")]:
    obj = 0.5 * (ws2 - v0) ** 2 + lm * np.abs(ws2)
    ax[1].plot(ws2, obj, lw=2.0, color=col, label=f"$\\lambda$ = {lm:g}")
    wmin = ws2[np.argmin(obj)]
    ax[1].plot(wmin, obj.min(), "o", color=col, ms=9, mec="k", mew=0.7, zorder=6)
ax[1].axvline(0, color="0.6", lw=0.9)
ax[1].annotate("for $\\lambda \\geq 3$ the minimum\nsits AT the kink, $w = 0$",
               xy=(0, 0.5 * v0 ** 2), xytext=(40, -26), textcoords="offset points",
               fontsize=8, color="crimson",
               bbox=dict(fc="white", ec="crimson", lw=0.9, alpha=0.95),
               arrowprops=dict(arrowstyle="->", color="crimson", lw=1.2))
ax[1].set_xlabel("$w$")
ax[1].set_ylabel(r"$\frac{1}{2}(w - 3)^2 + \lambda|w|$")
ax[1].set_title("What the prox solves, for $v$ = 3", fontsize=9)
ax[1].legend(fontsize=8.5, loc="upper center")

plt.tight_layout()
plt.show()

**Figure 2 — the soft-threshold, and the problem it solves.**

*Left:* the prox operators as functions. The ridge prox scales everything by
$1/(1+t\lambda)$ — a line through the origin, so a non-zero input gives a non-zero output,
always. The soft-threshold slides the line down by $t\lambda$ on the right, up by
$t\lambda$ on the left, and **clamps the gap to zero**. The shaded band is the dead zone:
any $|v| \leq t\lambda$ comes out as exactly $0.0$, a floating-point zero you can count
with `np.count_nonzero`.

Notice too that outside the band, the two curves are parallel to the identity but offset —
lasso subtracts a *constant* $t\lambda$, while ridge subtracts a *proportion*. Large
coefficients are therefore biased downward by the same absolute amount, which is the
well-known price of lasso: correct support, slightly shrunken values.

*Right:* the one-dimensional problem the prox minimizes, for $v = 3$ and four values of
$\lambda$. As $\lambda$ grows the V-shaped penalty tips the parabola leftward and the
minimum slides toward zero — and at $\lambda = 3$ it arrives *at the kink* and stops
there. It cannot go further: the kink is a genuine minimum because the two one-sided
slopes have opposite signs. That is the clamping in the formula, seen as geometry, and it
is why the closed form matches the brute-force grid search exactly in the cell above.

---

## 3. Proximal gradient (ISTA)

Now put it together. Split the objective into the smooth part and the awkward part:

$$\min_w \; f(w) + r(w)$$

Take a **gradient step on $f$ only**, then apply the **prox of $r$**:

$$\boxed{\;w_{k+1} = \mathrm{prox}_{\alpha r}\big(w_k - \alpha \nabla f(w_k)\big)\;}, \qquad \alpha \le 1/L$$

With $r = \lambda\|\cdot\|_1$ this is **ISTA** (iterative shrinkage-thresholding). Each
iteration is: descend, then shrink-and-clamp. Notice the step is *identical* to gradient
descent except for the prox wrapped around it — which is why the optimizer reuses
`GLMLoss` untouched.

The condition $\alpha \le 1/L$ is exactly day 2's descent lemma. Nothing about the
smooth part changed.

> **Design note.** `ProximalGradient` requires only `Regularizer.prox`. It never calls
> `gradient` on the penalty — which is why `L1.gradient` is free to **raise**. That is not
> a hole in the implementation; it is the contract, and the interface is segregated
> precisely so that an honest refusal costs nothing.

**FISTA** adds Nesterov momentum on an extrapolated point and improves the rate from
$O(1/k)$ to $O(1/k^2)$ — the same "momentum buys you a square root" bargain as day 2.

### Lasso is composition, not new machinery

$$\texttt{lasso} = \texttt{ProximalGradient(smooth=GLMLoss(X, y, SquaredError()), reg=L1(lam))}$$

`GLMLoss` is day 1's class, unchanged. A brand-new estimator, and not one existing file
was edited. That is the open/closed principle paying out for the last time this week.

In [ ]:
# Sparse ground truth: only 5 of 60 coefficients are non-zero.
n, p, k = 200, 60, 5
X = rng.normal(size=(n, p))
w_true = np.zeros(p)
w_true[rng.choice(p, k, replace=False)] = rng.normal(size=k) * 3
y = X @ w_true + 0.1 * rng.normal(size=n)

L = np.linalg.eigvalsh(X.T @ X / n).max()        # smoothness constant of the smooth part
grad_f = lambda w: X.T @ (X @ w - y) / n


def ista(lam, n_iter=4000, accelerated=False):
    w = z = np.zeros(p)
    t = 1.0
    alpha = 1.0 / L
    for _ in range(n_iter):
        w_new = soft_threshold(z - alpha * grad_f(z), alpha * lam)
        if accelerated:
            t_new = (1 + np.sqrt(1 + 4 * t * t)) / 2
            z = w_new + ((t - 1) / t_new) * (w_new - w)
            t = t_new
        else:
            z = w_new
        w = w_new
    return w


w_lasso = ista(0.1)
w_ridge = np.linalg.solve(X.T @ X / n + 0.1 * np.eye(p), X.T @ y / n)

print(f"true non-zeros  : {np.count_nonzero(w_true)} of {p}")
print(f"lasso non-zeros : {np.count_nonzero(w_lasso)} of {p}   <- EXACT zeros")
print(f"ridge non-zeros : {np.count_nonzero(w_ridge)} of {p}   <- nothing is ever exactly 0")
print(f"\nridge smallest |coefficient| = {np.abs(w_ridge).min():.3e}  (small, but not zero)")
print(f"support recovered correctly     : "
      f"{set(np.flatnonzero(w_lasso)) == set(np.flatnonzero(w_true))}")

In [ ]:
# The regularization path: sparsity as a function of lambda.
print(f"{'lambda':>8} {'non-zeros':>11} {'||w||_1':>10}")
for lam in [0.002, 0.004, 0.008, 0.016, 0.032, 0.064, 0.5, 2.0, 5.0]:
    w = ista(lam, n_iter=3000)
    print(f"{lam:8.3f} {np.count_nonzero(w):11d} {np.abs(w).sum():10.3f}")
print("\nIncreasing lambda switches coefficients off, and the order in which they go")
print("is a feature-importance ranking produced by the optimizer itself. Push lambda")
print("high enough and even the real ones die: the far end of the path is all bias.")

In [ ]:
# The regularization path: every coefficient, as lambda sweeps.
# Go down to a much smaller lambda, so the spurious coefficients actually appear.
lams = np.geomspace(1e-4, 5.0, 55)
paths = np.array([ista(lm, n_iter=2500) for lm in lams])
true_idx = set(np.flatnonzero(w_true))

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.4),
                       gridspec_kw={"width_ratios": [1.25, 1]})

for j in range(p):
    is_true = j in true_idx
    ax[0].semilogx(lams, paths[:, j],
                   color="crimson" if is_true else "tab:blue",
                   lw=2.2 if is_true else 1.0,
                   zorder=5 if is_true else 2, alpha=1.0 if is_true else 0.55)
ax[0].axhline(0, color="k", lw=1.0)
ax[0].plot([], [], color="crimson", lw=2.2, label=f"the {len(true_idx)} truly non-zero")
ax[0].plot([], [], color="tab:blue", lw=1.0, alpha=0.55,
           label=f"the {p - len(true_idx)} true zeros")
if ok := [i for i in range(len(lams))
          if set(np.flatnonzero(paths[i])) == true_idx]:
    ax[0].axvspan(lams[ok[0]], lams[ok[-1]], color="tab:green", alpha=0.13, zorder=0)
    ax[0].annotate("exact support", xy=(np.sqrt(lams[ok[0]] * lams[ok[-1]]), 0.3),
                   fontsize=8, color="darkgreen", ha="center",
                   bbox=dict(fc="white", ec="darkgreen", lw=0.8, alpha=0.95))
# Symlog: linear inside +-1e-3 (so exact zeros sit on the axis), logarithmic outside.
# Without this the 55 spurious coefficients are invisible against the +-5 real ones.
ax[0].set_yscale("symlog", linthresh=1e-3, linscale=0.5)
ax[0].set_ylim(-20, 20)
ax[0].set_xlabel(r"$\lambda$   (more regularization $\rightarrow$)")
ax[0].set_ylabel("coefficient value   (symlog: linear within $\\pm10^{-3}$)")
ax[0].set_title("The regularization path: noise dies first, signal dies last", fontsize=9)
ax[0].legend(fontsize=8.5, loc="lower left")

nz = np.count_nonzero(paths, axis=1)
ax[1].semilogx(lams, nz, "o-", color="tab:purple", lw=1.9, ms=4)
ax[1].axhline(len(true_idx), color="crimson", ls="--", lw=1.4)
ax[1].annotate(f"the true sparsity, {len(true_idx)}", xy=(lams[0] * 1.1, len(true_idx)),
               xytext=(0, 9), textcoords="offset points", fontsize=8, color="crimson")
if ok:
    ax[1].axvspan(lams[ok[0]], lams[ok[-1]], color="tab:green", alpha=0.16, zorder=0)
    ax[1].annotate("exactly the right\nfeatures chosen",
                   xy=(np.sqrt(lams[ok[0]] * lams[ok[-1]]), p * 0.62), fontsize=8,
                   color="darkgreen", ha="center",
                   bbox=dict(fc="white", ec="darkgreen", lw=0.9, alpha=0.95))
ax[1].set_xlabel(r"$\lambda$"); ax[1].set_ylabel("number of non-zero coefficients")
ax[1].set_title("Sparsity is a dial, and $\\lambda$ turns it", fontsize=9)

plt.tight_layout()
plt.show()

if ok:
    print(f"exact support recovery for lambda in [{lams[ok[0]]:.4f}, {lams[ok[-1]]:.4f}]")
print(f"at lambda = {lams[-1]:.2f}, {np.count_nonzero(paths[-1])} coefficients survive")

**Figure 3 — the regularization path is a feature-selection ranking.**

*Left:* every one of the 60 coefficients, traced as $\lambda$ sweeps from small to large.
The five genuinely non-zero coefficients are red; the 55 true zeros are grey. Read it
right to left, in the direction of decreasing regularization, and it is a story about
signal emerging from noise: at large $\lambda$ everything is dead, and as the penalty
relaxes the real coefficients come to life **first**, one at a time, in order of how
strongly the data supports them. The grey ones stay pinned at zero until $\lambda$ gets
genuinely small, and then they start to leak in — that is overfitting, visible as it
happens.

*Right:* the count of survivors. There is a whole **interval** of $\lambda$ (shaded) over
which lasso selects exactly the right five features — not a knife-edge, which is why the
method is usable in practice. Outside it you get too many (noise leaking in) or too few
(real signal suppressed).

> **The order in which coefficients switch on is produced by the optimizer itself**, and
> costs nothing beyond the path you were already computing. That is feature selection and
> model fitting done in a single sweep — and it is the practical reason lasso is
> everywhere, rather than the sparsity alone.

In [ ]:
# FISTA earns its keep only when the problem is badly conditioned -- on the easy
# design above, plain ISTA already finishes in a few dozen iterations.
Xc = rng.normal(size=(n, p)) + 3.0 * rng.normal(size=(n, 1))    # strongly correlated columns
yc = Xc @ w_true + 0.1 * rng.normal(size=n)
Lc = np.linalg.eigvalsh(Xc.T @ Xc / n).max()
print(f"condition number of this design: {np.linalg.cond(Xc.T @ Xc / n):.2e}")


def ista_c(lam, n_iter, accelerated=False):
    w = z = np.zeros(p)
    t, alpha = 1.0, 1.0 / Lc
    for _ in range(n_iter):
        w_new = soft_threshold(z - alpha * (Xc.T @ (Xc @ z - yc) / n), alpha * lam)
        if accelerated:
            t_new = (1 + np.sqrt(1 + 4 * t * t)) / 2
            z, t = w_new + ((t - 1) / t_new) * (w_new - w), t_new
        else:
            z = w_new
        w = w_new
    return w


def obj_c(w, lam):
    r = Xc @ w - yc
    return 0.5 * r @ r / n + lam * np.abs(w).sum()


lam = 0.05
best = obj_c(ista_c(lam, 200000, accelerated=True), lam)
print(f"\n{'iterations':>11} {'ISTA  f - f*':>16} {'FISTA  f - f*':>16} {'speedup':>10}")
for it in [100, 500, 2000, 10000]:
    a = obj_c(ista_c(lam, it), lam) - best
    b = obj_c(ista_c(lam, it, accelerated=True), lam) - best
    print(f"{it:11d} {a:16.3e} {b:16.3e} {a / b if b > 0 else float('inf'):10.1f}x")

In [ ]:
# ISTA against FISTA on the ill-conditioned design: the 1/k vs 1/k^2 difference.
iters = np.unique(np.geomspace(1, 20000, 45).astype(int))
gap_i, gap_f = [], []
w_i = z_f = w_f = np.zeros(p)
t_f, alpha_c = 1.0, 1.0 / Lc
g_c = lambda w: Xc.T @ (Xc @ w - yc) / n

prev = 0
for it in iters:
    for _ in range(it - prev):
        w_i = soft_threshold(w_i - alpha_c * g_c(w_i), alpha_c * lam)
        w_new = soft_threshold(z_f - alpha_c * g_c(z_f), alpha_c * lam)
        t_new = (1 + np.sqrt(1 + 4 * t_f * t_f)) / 2
        z_f, w_f, t_f = w_new + ((t_f - 1) / t_new) * (w_new - w_f), w_new, t_new
    prev = it
    gap_i.append(obj_c(w_i, lam) - best)
    gap_f.append(obj_c(w_f, lam) - best)

gap_i = np.maximum(np.array(gap_i), 1e-17)
gap_f = np.maximum(np.array(gap_f), 1e-17)

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.5))

# --- left: log-log, against the worst-case guarantees.
ax[0].loglog(iters, gap_i, "o-", color="crimson", lw=2.0, ms=4, label="ISTA")
ax[0].loglog(iters, gap_f, "s-", color="tab:blue", lw=2.0, ms=4, label="FISTA")
ax[0].loglog(iters, gap_i[0] / iters, ":", color="crimson", lw=1.6,
             label=r"$O(1/k)$ guarantee")
ax[0].loglog(iters, gap_f[0] / iters ** 2.0, ":", color="tab:blue", lw=1.6,
             label=r"$O(1/k^2)$ guarantee")
ax[0].annotate("both curves eventually fall\nFASTER than their guarantee",
               xy=(6000, 1e-9), xytext=(-40, 56), textcoords="offset points",
               fontsize=8, ha="center",
               bbox=dict(fc="white", ec="0.4", lw=0.9, alpha=0.95),
               arrowprops=dict(arrowstyle="->", lw=1.1))
ax[0].set_xlabel("iteration $k$"); ax[0].set_ylabel(r"$f(w_k) - f^\star$")
ax[0].set_title("Log–log: the guarantees are upper bounds, not predictions", fontsize=9)
ax[0].legend(fontsize=8)

# --- right: semilogy, where straight lines mean linear (geometric) convergence.
ax[1].semilogy(iters, gap_i, "-", color="crimson", lw=2.2, label="ISTA")
ax[1].semilogy(iters, gap_f, "-", color="tab:blue", lw=2.2, label="FISTA")
rows = []
for tol, col in [(1e-2, "0.35"), (1e-6, "0.35")]:
    ii = np.flatnonzero(gap_i < tol)
    ff = np.flatnonzero(gap_f < tol)
    if ii.size and ff.size:
        ki, kf = iters[ii[0]], iters[ff[0]]
        rows.append((tol, ki, kf))
        ax[1].axhline(tol, color=col, ls=":", lw=1.1)
        ax[1].annotate(f"tol {tol:.0e}:  FISTA {kf},  ISTA {ki}  ({ki/kf:.1f}x fewer)",
                       xy=(11800, tol), xytext=(0, 6), textcoords="offset points",
                       fontsize=8, color="0.2", ha="right",
                       bbox=dict(fc="white", ec="none", alpha=0.85, pad=1.0))
ax[1].set_xlabel("iteration $k$   (linear axis)")
ax[1].set_ylabel(r"$f(w_k) - f^\star$")
ax[1].set_title("Semilog: straight lines = linear convergence", fontsize=9)
ax[1].set_xlim(0, 12000); ax[1].legend(fontsize=8.5, loc="upper right")

plt.tight_layout()
plt.show()

print(f"{'tolerance':>12} {'ISTA':>8} {'FISTA':>8} {'speedup':>9}")
for tol in [1e-2, 1e-4, 1e-6, 1e-9]:
    ii, ff = np.flatnonzero(gap_i < tol), np.flatnonzero(gap_f < tol)
    if ii.size and ff.size:
        print(f"{tol:12.0e} {iters[ii[0]]:8d} {iters[ff[0]]:8d} "
              f"{iters[ii[0]] / iters[ff[0]]:8.1f}x")

**Figure 4 — the same bargain as day 2, and the same honest caveat.**

FISTA reaches any given accuracy several times faster than ISTA: about $9\times$ at
$10^{-2}$, still over $2\times$ at $10^{-9}$. One extra stored vector and an extrapolation
step, exactly the deal heavy ball offered on day 2.

But read the left panel carefully, because **the measured curves do not match the
advertised rates.** The dotted lines are the $O(1/k)$ and $O(1/k^2)$ guarantees, and both
methods eventually fall *far* below them. The right panel says why: on a semilog axis both
curves become straight, which means the convergence is ultimately **linear** (a constant
factor per iteration), not sublinear at all.

That is not a bug in the theory. $O(1/k)$ and $O(1/k^2)$ are worst-case bounds over all
convex problems of this shape. This particular problem is kinder: once ISTA has settled on
the correct support, the objective restricted to those five coordinates is a strongly
convex quadratic, and both methods inherit the geometric rate that comes with it.

> **A rate is a guarantee, not a prediction** — the same lesson as day 2's heavy ball,
> where the measured 93 steps beat the asymptotic 69, and day 4's Newton, whose quadratic
> convergence only appears at the end. Bounds tell you what cannot go wrong. They do not
> tell you what will happen.

Note also that FISTA is **not monotone** — its objective can rise on individual iterations,
visible as small bumps — because the extrapolated point $z_k$ is deliberately allowed to
overshoot. An optimizer that asserts the loss decreased every step will fire spuriously
here.

> **Why the correlated design?** On the tidy design used earlier in this section, plain
> ISTA converges in a few dozen iterations and there is simply no room for FISTA to show
> a benefit. Acceleration matters when $\kappa$ is large — exactly as on day 2. If you
> benchmark an accelerated method on an easy problem and conclude it does not help, the
> problem was the benchmark.

---

## 4. Robustness: the Huber loss

A different weakness, with a different fix. Day 1 derived squared error from a Gaussian
assumption — and Gaussian tails are very thin, so the model considers a far-out point
essentially impossible. Confronted with one anyway, the fit distorts itself to reduce that
one enormous residual.

The cause is visible in the derivative: for squared error $\varphi'(z,y) = z - y$ grows
**without bound**, so one sample's influence on the gradient is unlimited.

**Huber** caps it:

$$\varphi_\delta(r) = \begin{cases}
\tfrac{1}{2}r^2 & |r| \le \delta \\[4pt]
\delta\left(|r| - \tfrac{\delta}{2}\right) & |r| > \delta
\end{cases}
\qquad
\varphi_\delta'(r) = \begin{cases} r & |r| \le \delta \\ \delta\,\mathrm{sign}(r) & |r| > \delta\end{cases}$$

Quadratic near zero (so it behaves like least squares on the bulk of the data), linear
far out (so the derivative saturates at $\pm\delta$ and no single point can dominate).
The constants are chosen to make $\varphi$ and $\varphi'$ continuous at $|r| = \delta$.

### Two kinds of non-smoothness — do not confuse them

This is the distinction worth leaving the week with:

| | where | smooth? | what it does |
|---|---|---|---|
| **L1 penalty** | in the *parameters* $w$ | **no** — kink at $w_j = 0$ | creates sparsity; needs a prox |
| **Huber loss** | in the *residual* $r$ | **yes** — $\varphi'$ is continuous | bounds influence; ordinary gradient methods work |

Huber is robust *and* differentiable. It needs no new optimizer at all — it is just
another `PointwiseLoss`, so gradient descent, Newton and ridge accept it immediately.

In [ ]:
def huber(r, d):
    a = np.abs(r)
    return np.where(a <= d, 0.5 * r ** 2, d * (a - 0.5 * d))

def huber_d1(r, d):
    return np.clip(r, -d, d)

# As delta -> infinity Huber must become squared error.
r = np.linspace(-3, 3, 7)
print("delta -> inf recovers least squares:")
for d in [1.0, 10.0, 1e6]:
    print(f"  delta={d:8.0e}  max|huber - 0.5r^2| = {np.abs(huber(r, d) - 0.5 * r ** 2).max():.3e}")

print("\ninfluence of a single residual (the derivative):")
for rv in [1.0, 10.0, 1000.0]:
    print(f"  residual {rv:8.1f}  ->  squared error: {rv:10.1f}   huber(d=1): {huber_d1(rv, 1.0):6.1f}")
print("\nSquared error lets one point shout as loudly as it likes. Huber caps the volume.")

In [ ]:
# One gross outlier against a clean linear fit.
t = np.linspace(0, 10, 40)
y_clean = 2.0 * t + 1.0 + 0.3 * rng.normal(size=t.size)
y_bad = y_clean.copy()
y_bad[20] += 60.0                                   # one corrupted measurement

A = np.column_stack([t, np.ones_like(t)])

def fit_huber(d, n_iter=3000, lr=0.01):
    w = np.zeros(2)
    for _ in range(n_iter):
        w = w - lr * (A.T @ huber_d1(A @ w - y_bad, d) / len(t))
    return w

ls_clean = np.linalg.lstsq(A, y_clean, rcond=None)[0]
ls_bad = np.linalg.lstsq(A, y_bad, rcond=None)[0]
hu_bad = fit_huber(1.0)

print(f"{'fit':>26} {'slope':>9} {'intercept':>11}")
print(f"{'truth':>26} {2.0:9.3f} {1.0:11.3f}")
print(f"{'least squares, clean data':>26} {ls_clean[0]:9.3f} {ls_clean[1]:11.3f}")
print(f"{'least squares, 1 outlier':>26} {ls_bad[0]:9.3f} {ls_bad[1]:11.3f}   <- dragged off")
print(f"{'huber, same 1 outlier':>26} {hu_bad[0]:9.3f} {hu_bad[1]:11.3f}   <- barely moved")

In [ ]:
# Huber: the loss, its influence function, and what that does to a fit.
fig, ax = plt.subplots(1, 3, figsize=(14, 4.0))
rr = np.linspace(-4, 4, 800)
d = 1.0

ax[0].plot(rr, 0.5 * rr ** 2, lw=2.3, color="crimson", label=r"squared error $\frac{1}{2}r^2$")
ax[0].plot(rr, huber(rr, d), lw=2.3, color="tab:blue", label=f"Huber, $\\delta$ = {d}")
ax[0].plot(rr, np.abs(rr), "--", lw=1.5, color="0.55", label=r"absolute $|r|$")
ax[0].axvspan(-d, d, color="tab:blue", alpha=0.10)
ax[0].annotate("quadratic here", xy=(0, 0.25), xytext=(0, 44), textcoords="offset points",
               fontsize=8, color="tab:blue", ha="center",
               arrowprops=dict(arrowstyle="->", color="tab:blue", lw=1.1))
ax[0].set_xlabel("residual $r$"); ax[0].set_ylabel("loss")
ax[0].set_ylim(-0.3, 5); ax[0].set_title("The loss", fontsize=9)
ax[0].legend(fontsize=8)

ax[1].plot(rr, rr, lw=2.3, color="crimson", label=r"squared error: $r$  (unbounded)")
ax[1].plot(rr, huber_d1(rr, d), lw=2.6, color="tab:blue",
           label=r"Huber: $\mathrm{clip}(r, -\delta, \delta)$")
for s in (-1, 1):
    ax[1].axhline(s * d, color="tab:blue", ls=":", lw=1.2)
ax[1].annotate("capped at $\\pm\\delta$: no single point\ncan shout louder than this",
               xy=(2.6, d), xytext=(-30, 34), textcoords="offset points", fontsize=8,
               color="tab:blue", ha="center",
               bbox=dict(fc="white", ec="tab:blue", lw=0.9, alpha=0.95),
               arrowprops=dict(arrowstyle="->", color="tab:blue", lw=1.2))
ax[1].axhline(0, color="0.6", lw=0.8)
ax[1].set_xlabel("residual $r$"); ax[1].set_ylabel(r"influence $\varphi'(r)$")
ax[1].set_title("The influence function — where robustness comes from", fontsize=9)
ax[1].legend(fontsize=8, loc="upper left")

ax[2].plot(t, y_bad, "o", color="k", ms=5, zorder=5, label="data")
tl = np.array([t.min(), t.max()])
ax[2].plot(tl, 2.0 * tl + 1.0, "--", color="0.45", lw=2.0, label="truth")
ax[2].plot(tl, ls_bad[0] * tl + ls_bad[1], color="crimson", lw=2.4,
           label=f"least squares (intercept {ls_bad[1]:.2f})")
ax[2].plot(tl, hu_bad[0] * tl + hu_bad[1], color="tab:blue", lw=2.4,
           label=f"Huber (intercept {hu_bad[1]:.2f})")
# The outlier is far off the top of this zoom; mark where it went instead of
# rescaling, which would squash the two fitted lines together and hide the point.
ax[2].set_ylim(-1, 26)
ax[2].annotate(f"the corrupted point is off\nthe top, at y = {y_bad[20]:.0f}",
               xy=(t[20], 25.4), xytext=(58, -120), textcoords="offset points",
               fontsize=8, color="crimson", ha="left",
               bbox=dict(fc="white", ec="crimson", lw=0.9, alpha=0.95),
               arrowprops=dict(arrowstyle="-|>", color="crimson", lw=1.6,
                               connectionstyle="arc3,rad=0.2"))
ax[2].set_xlabel("$t$"); ax[2].set_ylabel("$y$")
ax[2].set_title(f"40 good points, 1 bad one (truth: intercept 1.00)", fontsize=9)
ax[2].legend(fontsize=7.5, loc="upper left")

plt.tight_layout()
plt.show()

**Figure 5 — robustness is a statement about the derivative.**

*(a)* The losses. Huber is exactly the parabola inside $|r| \leq \delta$ and switches to a
straight line outside — and the constants in the definition are precisely what makes the
join seamless in both value and slope.

*(b)* The derivative, which is the part that matters. Under squared error a residual of
$1000$ contributes $1000$ to the gradient: **one bad measurement can outvote hundreds of
good ones**, because nothing bounds its influence. Huber clips that contribution at
$\pm\delta$. The far-out point still says "I am above the line", it simply cannot say it
any louder than a point at distance $\delta$ can.

*(c)* The consequence, zoomed in on the honest data — the corrupted point is far above the
top of the frame. One bad value out of forty lifts the least-squares intercept from a true
$1.00$ to $2.29$, while Huber returns $0.93$, indistinguishable from the fit you would get
if the corruption had never happened.

> **And the crucial architectural point:** panel (b) is a *continuous* function. Huber is
> robust **and** differentiable, so it needs no prox, no new optimizer, nothing — it is
> one more `PointwiseLoss`, and every optimizer you wrote this week accepts it
> immediately. Contrast §1's $\ell_1$, whose non-smoothness is in the *parameters* and
> cost us an entire new algorithm. Two kinds of kink, two completely different bills.

One corrupted point out of forty visibly moves the least-squares line. Huber, seeing the
same data, barely notices — because that point's contribution to the gradient was capped
at $\delta$.

---

## 5. The week, in one diagram

Everything you built plugs into everything else, and nothing needed rewriting:

```
Objective ─────────────┬─ Quadratic, Rosenbrock
                       └─ GLMLoss(X, y, PointwiseLoss) ← SquaredError, LogisticNLL,
                              │                            Huber, PoissonNLL
                              │
        RegularizedObjective(·, Regularizer) ← L2  ....... ridge, for free
                              │
   DescentOptimizer(DirectionRule, LineSearch, Stopping, Observers)
        ├─ SteepestDescent + Armijo .......... gradient descent   (day 2)
        ├─ HeavyBall       + FixedStep ....... momentum           (day 2)
        └─ NewtonDirection(CholeskySolver) ... Newton / IRLS      (day 4)

   SGD, Adam  over BatchObjective ............ stochastic         (day 3)
   GaussNewton / LevenbergMarquardt(CholeskySolver)
                over LeastSquaresProblem ..... curve fitting      (day 5)
   ProximalGradient(GLMLoss, L1) ............. lasso              (day 6)
```

Count what was reused rather than rewritten: **one** descent loop, **one** Cholesky
solver (Newton, Gauss–Newton, LM), **one** GLM class (five losses), **one** regularizer
interface (four penalties). The architecture was not decoration — it is why day 6 costs
an afternoon instead of a week.

---

## 6. Today's labs

| Lab | What | The point |
|---|---|---|
| 1 (65 min) | `L1`, `ElasticNet`, `ProximalGradient` (ISTA, FISTA optional) | lasso reuses `GLMLoss` **unchanged** |
| 2 (40 min) | `Huber`, `PoissonNLL` | two new losses, and every optimizer accepts them for free |
| 3 (40 min) | benchmark → CSV → notebook | the one-pager: which optimizer for which problem |
| 4 (35 min) | **extensibility challenge** | add one implementation, editing **no** existing `src/` file (checked with `git diff`) |

The challenge in Lab 4 is the real exam for the week's architecture. `FISTA`,
`GroupLasso`, `Nesterov` or `PoissonNLL` — pick one and add it without touching anything
that exists. If you are forced to edit, say in `DESIGN.md` exactly which interface was too
narrow and what you would change. **An honest account of a design failure is worth more
than a lucky success.**

**Three questions for the debrief:**

1. Why does L1 create sparsity when ridge does not? Answer in terms of the derivative as
   $w \to 0$.
2. What is a proximal operator, and why does having one let you skip the gradient?
3. Why is Huber robust yet smooth — and how is its non-smoothness story different from L1's?

### Closing

You have written, from scratch, the optimizers behind linear and logistic regression,
neural-network training, curve fitting, and sparse model selection. More usefully, you can
now say for any new problem: *is it smooth? is it convex? is it a finite sum? is it a sum
of squares? is the penalty differentiable?* — and each answer names the method.

> The one habit worth keeping: **check the gradient, then try to break the method.**
> Everything you understand deeply this week, you understand because you watched it fail
> first.